In [ ]:
"""Kaggle T4 x2 / ComfyUI MiniMax-H3：整段粘贴到一个 Python 单元运行。2026111701
启用 Settings > Accelerator > GPU T4 x2、Internet。
采用 FL2VA pruned Q4_K_M + H3 专用 Qwen3-VL NVFP4 AWQ，原生 H3 视频/音频节点。
官方 T2V/I2V 模板已内嵌，并额外派生 T2I（复用同一套权重，只输出单帧 PNG，不下载新模型）与
IV2V（原片 + 人脸参考图 → 换脸短视频，改用 H3 ref2va 参考节点，同样不下载任何新模型）；
视频模型使用 GGUF 加载器，文本编码器使用原生 CLIPLoader，并将文本编码器指定到第二张 GPU。
这是跨卡分配组件，不是显存合并。
多参考（R2V）另外使用 ref2va pruned Q4_K_M + ref2v turbo 4-step LoRA，与 fl2va 并存，只影响多图 / 多影工作流。没有实测当前 Kaggle 的生成速度和峰值内存。
IT2V 改为单图参考（ref2va）：参考图只提供身份 / 外观，不当首帧，首帧由模型生成。
"""
import base64
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zlib

# ==================== 配置 ====================
ROOT = Path("/tmp/ComfyUI")
PORT = 8188
LOCAL_URL = f"http://127.0.0.1:{PORT}"
USE_SECOND_GPU = True        # 2 卡：DiT / VAE 默认 GPU0，文本编码器 GPU1
DURATION = 2.0               # 首轮先测试短片；实际帧数按官方 17k+5 网格取整
MEGAPIXELS = 0.1             # 低分辨率试跑，成功后逐步改 0.2 / 0.4；非质量推荐值
TURBO_STEPS = 4
# IT2V（单图参考）单独一组：默认优先 identity，不是速度。
# ref_image_size："max" 用 2048 短边，最贴脸；"match" 缩到生成面积，省显存但容易跑脸。
IT2V_REF_IMAGE_SIZE = "max"
IT2V_MEGAPIXELS = 0.4        # 0.1 只够测速；0.4 起脸才有细节
IT2V_TURBO = False           # False = 20 步、不挂 turbo LoRA；True = 4 步快速预览
UPDATE_CODE = False           # 默认复用已安装版本；新环境仍自动克隆，需更新时改 True
START_TUNNEL = True
# 工作流输出文件名：模板键 -> 文件名后缀（内嵌 i2v 模板输出为 IT2V）
WORKFLOW_NAMES = {"t2v": "T2V", "i2v": "IT2V"}
PROMPT = ("Cinematic live-action shot. A small orange cat walks slowly on a rainy street, "
          "neon lights reflecting in puddles. Slow tracking camera, natural motion. "
          "Audio: gentle rain and quiet footsteps. No subtitles or watermarks.")
T2I_PROMPT = ("Cinematic still frame: a small orange cat sitting on a rainy neon-lit street, "
              "puddles reflecting the signs, shallow depth of field, natural rim light, "
              "frozen moment, no motion blur, no text or watermarks.")
T2I_DURATION = 5 / 24        # T2I 只采 H3 允许的最短 5 帧（17k+5 网格），再取第一帧当静态图
IV2V_REF_FRAMES = 124      # IV2V 取原片前 124 帧（24fps 约 5s）当动作参考；输出更短时会被自动截断
IV2V_PROMPT = ("Cinematic live-action shot, same camera, framing and timing as <Video 1>. "
               "The person in <Video 1> keeps exactly the same motion, gestures, head movement and lip sync, "
               "but the face and identity come from <Picture 1>: same facial features, hairstyle, skin tone and age. "
               "Stable identity in every frame, no morphing, no flicker, no double face. "
               "Keep the background, wardrobe colors and lighting of <Video 1> unchanged. "
               "Audio: keep the voice, ambience and rhythm of <Video 1>. No subtitles or watermarks.")


# ==================== 多参考 R2V（多图 + 多影 + 提示词） ====================
# 多参考需要 ref2va 专用 checkpoint；与 fl2va 并存，不覆盖、不删除既有文件。
MULTI_REF_IMAGES = 3          # 参考图槽位（官方上限 9）
MULTI_REF_VIDEOS = 2          # 参考影片槽位（官方上限 3，每段可带自己的音轨）
MULTI_REF_REF_FRAMES = 124    # 每段参考影片取前 124 帧（24fps 约 5.2 秒）
MULTI_REF_DIT = "minimax_h3_ref2va_pruned-Q4_K_M.gguf"
MULTI_REF_LORA = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
MULTI_REF_PROMPT = (
    "subject_definitions:\n"
    "<Picture 1> is the main character: keep the same face, hairstyle, skin tone and age.\n"
    "<Picture 2> is the wardrobe and color reference for <Picture 1>.\n"
    "<Picture 3> is the location and lighting reference.\n"
    "<Video 1> gives the camera move, framing and timing to follow; <Audio 1> is its soundtrack.\n"
    "<Video 2> gives the body motion and gesture rhythm to follow; <Audio 2> is its soundtrack.\n\n"
    "shot: <Picture 1> stands in the place shown in <Picture 3>, wearing <Picture 2>, "
    "moving with the same body motion as <Video 2> and the same camera path as <Video 1>. "
    "Stable identity in every frame, no morphing, no flicker, no double face. "
    "Keep the lighting and color palette of <Picture 3>.\n\n"
    "audio: blend the ambience of <Audio 1> and <Audio 2>, keep the voice and rhythm. "
    "No subtitles or watermarks."
)

# 单图参考（IT2V）用的提示词：参考图只提供身份 / 外观，第一帧由模型生成。
REF_IMAGE_PROMPT = (
    "subject_definitions:\n"
    "<Picture 1> is the only subject and the identity reference: same face, same hairstyle, "
    "same skin tone, same body type, same outfit colors and age. Exactly one person in frame.\n\n"
    "shot: single continuous shot, no cuts, no split screen, no character sheet, no multiple views, "
    "no duplicated body, no on-screen text. The first frame is generated by the model, not <Picture 1> "
    "itself. <Picture 1> walks slowly on a rainy neon-lit street, puddles reflecting the signs, "
    "slow tracking camera, natural motion. Identity stays stable in every frame: no morphing, "
    "no flicker, no face drift.\n\n"
    "audio: gentle rain and quiet footsteps, no music, no subtitles or watermarks."
)

MANIFEST = [{'repo': 'leejet/MiniMax-H3-GGUF', 'revision': 'd9c4c6312b4728a68a15a35626d84775a6523783', 'file': 'minimax_h3_fl2va_pruned-Q4_K_M.gguf', 'size': 11420663904, 'sha256': 'dd948e08ad0ba3c71bd42f368e283dd82e790f5122a63b276e22a3e0283d0c10', 'folder': 'diffusion_models'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'size': 15687142551, 'sha256': '35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6', 'folder': 'text_encoders'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'vae/minimax_h3_video_vae_fp16.safetensors', 'size': 5207808496, 'sha256': '7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522', 'folder': 'vae'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'vae/minimax_h3_audio_vae_fp32.safetensors', 'size': 605254808, 'sha256': '8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48', 'folder': 'vae'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors', 'size': 1956192992, 'sha256': 'c396a9a06f58399e9df9754b18299818d84a2ddd371724ba48fe4a41221437dc', 'folder': 'loras'}]

# ref2va 专用权重：只给多参考工作流使用，与上面的 fl2va 并存。
MANIFEST += [
    {'repo': 'leejet/MiniMax-H3-GGUF', 'revision': 'd9c4c6312b4728a68a15a35626d84775a6523783', 'file': 'minimax_h3_ref2va_pruned-Q4_K_M.gguf', 'size': 11420663904, 'sha256': '9ca1aef9f4609a56c2c751484013e1b20fe352f892122e5ab0a428fb95a7956e', 'folder': 'diffusion_models'},
    {'repo': 'Comfy-Org/MiniMax-H3', 'revision': '54472221d27c695fca9621b0c3ed52f06c4af707', 'file': 'loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors', 'size': 1956193000, 'sha256': '5b9ab5ade15d0775676d01a907268a69a1468dc6033b3b0d3ded5502f3ebb84c', 'folder': 'loras'},
]
TEMPLATE_BUNDLE = '...'


def run(*args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)


def tail(path):
    path = Path(path)
    if not path.exists():
        return "尚无日志"
    with path.open("rb") as f:
        f.seek(max(0, path.stat().st_size - 12000))
        return f.read().decode("utf-8", "replace")


def request_json(path, timeout=10):
    # 本地服务不经过 Notebook 的 HTTP_PROXY。
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    with opener.open(LOCAL_URL + path, timeout=timeout) as r:
        return json.load(r)


def ready():
    try:
        return "system" in request_json("/system_stats", timeout=2)
    except Exception:
        return False


def stop_owned(name):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)



def prepare_python_env(env_dir, base_python):
    """跳过 ensurepip，修复上次失败留下的半成品 venv，不删除已有依赖。"""
    env_dir = Path(env_dir)
    python = env_dir / "bin/python"
    # 每次重写解释器配置；不能只判断 bin/python 是否存在。
    run(base_python, "-m", "venv", "--without-pip", "--system-site-packages", env_dir)
    probe = subprocess.run([str(python), "-c",
        "import sys; print(sys.prefix); assert sys.prefix != sys.base_prefix"],
        capture_output=True, text=True)
    if probe.returncode or Path(probe.stdout.strip()).resolve() != env_dir.resolve():
        raise RuntimeError("虚拟环境解释器检查失败：" + probe.stderr[-3000:])
    pip_probe = subprocess.run([str(python), "-m", "pip", "--version"],
                               capture_output=True, text=True)
    if pip_probe.returncode:
        # pip 官方支持使用外部 pip 管理尚未安装 pip 的虚拟环境。
        manager = subprocess.run([str(base_python), "-m", "pip", "--version"],
                                 capture_output=True, text=True)
        if manager.returncode:
            raise RuntimeError("Notebook 的 Python 也无法调用 pip：" + manager.stderr[-3000:] +
                               "\n请在 Notebook 运行 %pip --version，提供完整输出。")
        result = subprocess.run([str(base_python), "-m", "pip", "--python", str(python),
                                 "install", "pip"], capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("用外部 pip 初始化虚拟环境失败：\n" +
                               result.stdout[-2000:] + result.stderr[-4000:])
        run(python, "-m", "pip", "--version")
    else:
        print(pip_probe.stdout.strip(), flush=True)
    return python


def clone_or_update(url, folder):
    folder = Path(folder)
    if not folder.exists():
        run("git", "clone", "--depth", "1", url, folder)
    elif not (folder / ".git").is_dir():
        raise RuntimeError(f"{folder} 已存在但不是 git 仓库；请换 ROOT 或先移开该目录。")
    elif UPDATE_CODE:
        changes = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=folder, text=True)
        if changes.strip():
            raise RuntimeError(f"{folder} 有本地代码改动，未覆盖。请保留改动后手工更新，或设置 UPDATE_CODE=False。")
        run("git", "pull", "--ff-only", cwd=folder)


def verify(path, item):
    path = Path(path)
    if not path.is_file() or path.stat().st_size != item["size"]:
        return False
    print("校验 SHA-256：", path.name, flush=True)
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest() == item["sha256"]


def download_model(item):
    target = ROOT / "models" / item["folder"] / Path(item["file"]).name
    target.parent.mkdir(parents=True, exist_ok=True)
    if verify(target, item):
        print("复用：", target.name, flush=True)
        return
    temporary = target.with_name(target.name + ".part")
    url = f"https://huggingface.co/{item['repo']}/resolve/{item['revision']}/{item['file']}"
    for attempt in range(1, 4):
        try:
            offset = temporary.stat().st_size if temporary.exists() else 0
            if offset >= item["size"]:
                if verify(temporary, item):
                    temporary.replace(target)
                    return
                temporary.unlink()
                offset = 0
            headers = {"User-Agent": "Kaggle-ComfyUI-H3/1.0"}
            if offset:
                headers["Range"] = f"bytes={offset}-"
            print(f"下载 {target.name}，已完成 {offset / 2**30:.2f} GiB，尝试 {attempt}/3", flush=True)
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120) as r:
                append = offset > 0 and r.status == 206
                if append and not r.headers.get("Content-Range", "").startswith(f"bytes {offset}-"):
                    raise RuntimeError("服务端断点位置不符")
                if r.status not in (200, 206):
                    raise RuntimeError(f"HTTP {r.status}")
                if r.status == 206 and not append:
                    raise RuntimeError("意外的部分文件响应")
                received = offset if append else 0
                last_print = time.monotonic()
                with temporary.open("ab" if append else "wb") as f:
                    while True:
                        chunk = r.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        received += len(chunk)
                        if received > item["size"]:
                            raise RuntimeError("下载大小超过预期")
                        f.write(chunk)
                        if time.monotonic() - last_print > 20:
                            print(f"  {received / 2**30:.2f} / {item['size'] / 2**30:.2f} GiB", flush=True)
                            last_print = time.monotonic()
            if not verify(temporary, item):
                if temporary.stat().st_size == item["size"]:
                    temporary.unlink()  # 完整但摘要错误的文件不能续传
                raise RuntimeError("文件长度或 SHA-256 不符")
            temporary.replace(target)
            return
        except Exception as e:
            print(f"下载失败：{type(e).__name__}: {e}", flush=True)
            if attempt == 3:
                raise RuntimeError(f"下载 {target.name} 失败；保留断点，下次重跑可续传。") from e
            time.sleep(2 * attempt)


def iter_graphs(workflow):
    yield workflow
    for graph in workflow.get("definitions", {}).get("subgraphs", []):
        yield graph


def set_widget(node, key, value):
    named = node.get("widgets_values_named", {})
    if key not in named:
        return
    keys = list(named)
    index = keys.index(key)
    if index < len(node.get("widgets_values", [])):
        node["widgets_values"][index] = value
    named[key] = value


def prune_dead_nodes(workflow):
    """清掉顶层输入空接、输出无人使用的辅助节点：它们从不进入执行链，只会干扰阅读。"""
    terminal = {"SaveVideo", "SaveImage", "PreviewImage", "PreviewAudio", "MarkdownNote"}
    while True:
        used = {link[1] for link in workflow["links"]}
        dead = {n["id"] for n in workflow["nodes"]
                if n.get("inputs") and n["type"] not in terminal and n["id"] not in used}
        if not dead:
            return workflow
        workflow["nodes"] = [n for n in workflow["nodes"] if n["id"] not in dead]
        workflow["links"] = [l for l in workflow["links"] if l[1] not in dead and l[3] not in dead]


def patch_workflow(workflow, second_gpu):
    replacements = {
        "minimax_h3_fl2va_pruned_int8_convrot.safetensors": MANIFEST[0]["file"],
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": Path(MANIFEST[1]["file"]).name,
        "minimax_h3_video_vae_int8_convrot.safetensors": Path(MANIFEST[2]["file"]).name,
        "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors": Path(MANIFEST[4]["file"]).name,
    }
    # 同时更新暴露控件、子图控件、模型下载元数据和模型名称枚举。
    def replace(value):
        if isinstance(value, str):
            for old, new in replacements.items():
                if value == old:
                    return new
            return value
        if isinstance(value, list):
            return [replace(v) for v in value]
        if isinstance(value, dict):
            return {k: replace(v) for k, v in value.items()}
        return value
    workflow = replace(workflow)
    for graph in iter_graphs(workflow):
        clip_nodes = []
        for node in graph["nodes"]:
            kind = node["type"]
            named = node.get("widgets_values_named", {})
            if kind == "UNETLoader":
                node["type"] = "UnetLoaderGGUF"
                node["widgets_values"] = [MANIFEST[0]["file"]]
                node["widgets_values_named"] = {"unet_name": MANIFEST[0]["file"]}
                item = MANIFEST[0]
            elif kind == "CLIPLoader":
                node["type"] = "CLIPLoader"
                node["widgets_values"] = [Path(MANIFEST[1]["file"]).name, "minimax", "default"]
                node["widgets_values_named"] = {"clip_name": Path(MANIFEST[1]["file"]).name, "type": "minimax", "device": "default"}
                clip_nodes.append(node)
                item = MANIFEST[1]
            else:
                item = None
            if item:
                node["properties"] = {"Node name for S&R": node["type"], "models": [{
                    "name": Path(item["file"]).name, "directory": item["folder"],
                    "url": f"https://huggingface.co/{item['repo']}/resolve/{item['revision']}/{item['file']}"}]}
            elif kind == "VAELoader":
                node.get("properties", {}).pop("models", None)
            elif kind == "LoraLoaderModelOnly":
                node.get("properties", {}).pop("models", None)
            if kind == "ResolutionSelector":
                set_widget(node, "megapixels", MEGAPIXELS)
            # 修改主子图暴露的控件，保留它们原本的索引和图连线。
            if "unet_name" in named and "prompt" in named:
                set_widget(node, "prompt", PROMPT)
                set_widget(node, "value_1", DURATION)
                set_widget(node, "value", True)
                set_widget(node, "value_2", TURBO_STEPS)
            if kind == "PrimitiveFloat" and "duration" in node.get("title", ""):
                set_widget(node, "value", DURATION)
            if kind == "PrimitiveBoolean":
                set_widget(node, "value", True)
            if kind == "PrimitiveInt" and node.get("inputs"):
                if any(i.get("name") == "value" and i.get("link") is not None for i in node["inputs"]):
                    set_widget(node, "value", TURBO_STEPS)
            if kind == "MarkdownNote" and node.get("title") == "Note: Model Links":
                text = ("Kaggle H3：FL2VA pruned Q4_K_M + 专用 Qwen3-VL NVFP4 AWQ。\n"
                        "GPU0：DiT/VAE；GPU1：文本编码器（需两卡可见）。双卡不是自动合并显存。\n"
                        "这是官方 H3 模板的 GGUF 适配副本；原生模板需要自行替换两个加载器。\n"
                        "先用低分辨率短片试跑，再提高分辨率和时长。图生视频请先上传自己的图片。")
                node["widgets_values"] = [text]
                node["widgets_values_named"] = {"text": text}
        if second_gpu and clip_nodes:
            # 子图采用对象形式的 links。只插入设备节点，不改 H3 算法或采样路径。
            if graph is workflow:
                raise RuntimeError("模板结构变化：加载器不在子图中")
            for clip in clip_nodes:
                outgoing = [l for l in graph["links"] if l["origin_id"] == clip["id"] and l["origin_slot"] == 0]
                node_id = max(n["id"] for n in graph["nodes"]) + 1
                link_id = max(l["id"] for l in graph["links"]) + 1
                for link in outgoing:
                    link["origin_id"] = node_id
                graph["links"].append({"id": link_id, "origin_id": clip["id"], "origin_slot": 0,
                    "target_id": node_id, "target_slot": 0, "type": "CLIP"})
                clip["outputs"][0]["links"] = [link_id]
                graph["nodes"].append({"id": node_id, "type": "SelectCLIPDevice", "pos": [-1450, 5150],
                    "size": [300, 90], "flags": {}, "order": 11, "mode": 0,
                    "inputs": [{"name": "clip", "type": "CLIP", "link": link_id}],
                    "outputs": [{"name": "CLIP", "type": "CLIP", "links": [l["id"] for l in outgoing]}],
                    "properties": {"Node name for S&R": "SelectCLIPDevice"},
                    "widgets_values": ["gpu:1"], "widgets_values_named": {"device": "gpu:1"},
                    "title": "H3 Text Encoder → GPU 1"})
                graph["state"]["lastNodeId"] = node_id
                graph["state"]["lastLinkId"] = link_id
    return workflow


def build_t2i_workflow(workflow, slice_first_frame):
    """由已适配的 T2V 模板派生 T2I：不下载任何新模型，只把输出从视频换成单帧 PNG。

    复用同一个 H3 子图（文本编码 + 采样 + 视频 VAE 解码），在子图末端多暴露一个 IMAGES 输出，
    顶层用 ImageFromBatch 取第一帧交给 SaveImage。权重、Turbo LoRA、第二张 GPU 上的文本编码器
    连线都与 T2V 完全一致，因此不会额外下载任何模型。
    """
    workflow = json.loads(json.dumps(workflow))
    subgraphs = workflow.get("definitions", {}).get("subgraphs", [])
    if len(subgraphs) != 1:
        raise RuntimeError(f"模板结构变化：预期 1 个 H3 子图，实际 {len(subgraphs)} 个")
    graph = subgraphs[0]
    if len(graph.get("outputs", [])) != 1 or graph["outputs"][0].get("type") != "VIDEO":
        raise RuntimeError("模板结构变化：H3 子图输出不是单一 VIDEO")
    decode = next((n for n in graph["nodes"] if n["type"] == "VAEDecode"), None)
    if decode is None or not decode.get("outputs"):
        raise RuntimeError("模板结构变化：子图里缺少 VAEDecode")

    # 1) 子图 links 是对象格式：VAEDecode.IMAGE → 子图新输出 IMAGES。
    sg_link_id = max([l["id"] for l in graph["links"]] + [0]) + 1
    output_slot = len(graph["outputs"])
    graph["links"].append({"id": sg_link_id, "origin_id": decode["id"], "origin_slot": 0,
                           "target_id": -20, "target_slot": output_slot, "type": "IMAGE"})
    decode["outputs"][0]["links"] = list(decode["outputs"][0].get("links") or []) + [sg_link_id]
    graph["outputs"].append({"id": f"{sg_link_id:08x}-0000-4000-8000-{sg_link_id:012x}", "name": "IMAGES", "type": "IMAGE",
                             "linkIds": [sg_link_id], "localized_name": "IMAGES", "pos": [2154, 5194]})
    state = graph.setdefault("state", {})
    state["lastLinkId"] = max(state.get("lastLinkId", 0), sg_link_id)

    # 2) 顶层 links 是数组格式：移除 SaveVideo，改成 IMAGES → (ImageFromBatch) → SaveImage。
    instance = next((n for n in workflow["nodes"] if n["type"] == graph["id"]), None)
    if instance is None:
        raise RuntimeError("模板结构变化：找不到 H3 子图实例节点")
    removed = {n["id"] for n in workflow["nodes"] if n["type"] in ("SaveVideo", "SaveImage", "ImageFromBatch")}
    workflow["nodes"] = [n for n in workflow["nodes"] if n["id"] not in removed]
    workflow["links"] = [l for l in workflow["links"] if l[1] not in removed and l[3] not in removed]
    instance["outputs"][0]["links"] = []
    instance["outputs"].append({"name": "IMAGES", "type": "IMAGE", "links": []})
    image_slot = len(instance["outputs"]) - 1

    next_node = max([workflow.get("last_node_id", 0)] + [n["id"] for n in workflow["nodes"]]) + 1
    next_link = max([workflow.get("last_link_id", 0)] + [l[0] for l in workflow["links"]]) + 1
    order = max([n.get("order", 0) for n in workflow["nodes"]] + [0]) + 1

    def link(origin_id, origin_slot, target_id, target_slot):
        nonlocal next_link
        workflow["links"].append([next_link, origin_id, origin_slot, target_id, target_slot, "IMAGE"])
        next_link += 1
        return next_link - 1

    save_image = {"id": next_node, "type": "SaveImage", "pos": [-140, 4840], "size": [520, 420],
                  "flags": {}, "order": order, "mode": 0,
                  "inputs": [{"name": "images", "type": "IMAGE", "link": None}],
                  "outputs": [], "properties": {"cnr_id": "comfy-core", "ver": "0.33.0"},
                  "widgets_values": ["image/MiniMax_H3"],
                  "widgets_values_named": {"filename_prefix": "image/MiniMax_H3"},
                  "title": "Save Image (first frame)"}
    workflow["nodes"].append(save_image)
    batch = None
    if slice_first_frame:
        batch = {"id": next_node + 1, "type": "ImageFromBatch", "pos": [-780, 4840], "size": [300, 110],
                 "flags": {}, "order": order + 1, "mode": 0,
                 "inputs": [{"name": "image", "type": "IMAGE", "link": None}],
                 "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": []}],
                 "properties": {"cnr_id": "comfy-core", "ver": "0.33.0"},
                 "widgets_values": [0, 1],
                 "widgets_values_named": {"batch_index": 0, "length": 1},
                 "title": "First frame only"}
        workflow["nodes"].append(batch)
    # 先连到 batch（只留第一帧），没有 batch 节点时直接连到 SaveImage。
    inst_link = link(instance["id"], image_slot, (batch or save_image)["id"], 0)
    instance["outputs"][image_slot]["links"].append(inst_link)
    if batch:
        batch["inputs"][0]["link"] = inst_link
        batch_link = link(batch["id"], 0, save_image["id"], 0)
        batch["outputs"][0]["links"].append(batch_link)
        save_image["inputs"][0]["link"] = batch_link
    else:
        save_image["inputs"][0]["link"] = inst_link
    workflow["last_node_id"] = max(n["id"] for n in workflow["nodes"])
    workflow["last_link_id"] = next_link - 1

    # 3) 静态图用固定提示词与最短时长；控件索引沿用 T2V 模板，不新增模型或节点依赖。
    set_widget(instance, "prompt", T2I_PROMPT)
    set_widget(instance, "value_1", T2I_DURATION)
    for note in workflow["nodes"]:
        if note.get("type") == "MarkdownNote" and note.get("title") == "Note: MiniMax H3":
            text = ("## MiniMax H3 · Kaggle T2I\n\n"
                    "本工作流由官方 T2V 模板派生，**不下载任何新模型**：沿用同一套 H3 权重、Turbo LoRA "
                    "与第二张 GPU 上的文本编码器。\n\n"
                    "- 子图仍按 H3 网格采样最短 5 帧，再用 ImageFromBatch 取第一帧存成 PNG。\n"
                    "- 想要更接近静态照片，请在 prompt 里写明 still frame / frozen moment，少写镜头运动。\n"
                    "- 需要整段画面与声音时改用 MiniMax_H3_Kaggle_T2V。\n")
            note["widgets_values"] = [text]
            note["widgets_values_named"] = {"text": text}
    return workflow


def build_ref_image_workflow(workflow):
    """IT2V：1 张参考图 + 提示词（ref2va），参考图不当首帧。

    由官方 i2v 模板派生：子图里的条件节点换成 MiniMaxH3ReferenceToVideo，
    只保留 ref_images.ref_image_0 一路参考图；first_frame / last_frame 连同
    对应的子图输入一起移除，所以生成视频的第一帧由模型自己产生，参考图只提供
    身份 / 外观。模型改用 ref2va pruned Q4_K_M + ref2v turbo 4-step LoRA，
    与 fl2va 并存，不覆盖既有文件。
    """
    workflow = json.loads(json.dumps(workflow))
    subgraphs = workflow.get("definitions", {}).get("subgraphs", [])
    if len(subgraphs) != 1:
        raise RuntimeError(f"模板结构变化：预期 1 个 H3 子图，实际 {len(subgraphs)} 个")
    graph = subgraphs[0]
    if len(graph.get("outputs", [])) != 1 or graph["outputs"][0].get("type") != "VIDEO":
        raise RuntimeError("模板结构变化：H3 子图输出不是单一 VIDEO")
    cond = next((n for n in graph["nodes"] if n["type"] == "MiniMaxH3ImageToVideo"), None)
    if cond is None:
        raise RuntimeError("模板结构变化：子图里缺少 MiniMaxH3ImageToVideo")
    instance = next((n for n in workflow["nodes"] if n["type"] == graph["id"]), None)
    if instance is None:
        raise RuntimeError("模板结构变化：找不到 H3 子图实例节点")
    audio_decode = next((n for n in graph["nodes"] if n["type"] == "VAEDecodeAudio"), None)
    audio_vae_link_in = next((l for l in graph["links"] if audio_decode is not None and
                              l["target_id"] == audio_decode["id"] and l["type"] == "VAE"), None)
    audio_vae = next((n for n in graph["nodes"] if audio_vae_link_in is not None and
                      n["id"] == audio_vae_link_in["origin_id"]), None)
    if audio_vae is None:
        raise RuntimeError("模板结构变化：找不到音频 VAE 加载器")

    old = {i["name"]: i.get("link") for i in cond["inputs"]}
    missing = {"clip", "vae", "first_frame", "last_frame", "prompt", "width", "height", "length"} - set(old)
    if missing:
        raise RuntimeError("模板结构变化：MiniMaxH3ImageToVideo 缺少输入 " + ", ".join(sorted(missing)))

    dit_item = next(item for item in MANIFEST if Path(item["file"]).name == MULTI_REF_DIT)
    lora_item = next(item for item in MANIFEST if Path(item["file"]).name == MULTI_REF_LORA)
    state = graph.setdefault("state", {})
    cond_id = cond["id"]
    next_node = max([state.get("lastNodeId", 0)] + [n["id"] for n in graph["nodes"]]) + 1
    next_link = max([state.get("lastLinkId", 0)] + [l["id"] for l in graph["links"]]) + 1

    def add_node(node):
        nonlocal next_node
        node["id"] = next_node
        next_node += 1
        graph["nodes"].append(node)
        return node

    def add_link(origin_id, origin_slot, target_id, target_slot, kind):
        nonlocal next_link
        graph["links"].append({"id": next_link, "origin_id": origin_id, "origin_slot": origin_slot,
                               "target_id": target_id, "target_slot": target_slot, "type": kind})
        next_link += 1
        return next_link - 1

    # 1) 音频 VAE 接上 ref2va 的 audio_vae：输出仍要带音轨。
    audio_vae_link = add_link(audio_vae["id"], 0, cond_id, 2, "VAE")
    audio_vae["outputs"][0]["links"] = list(audio_vae["outputs"][0].get("links") or []) + [audio_vae_link]

    # 2) 拆掉首尾帧：参考图不当首帧，第一帧交给模型生成。
    dropped = [link_id for link_id in (old["first_frame"], old["last_frame"]) if link_id is not None]
    graph["links"] = [l for l in graph["links"] if l["id"] not in dropped]
    removed = sorted(i for i, inp in enumerate(graph["inputs"])
                     if set(inp.get("linkIds") or []) & set(dropped))
    if len(removed) != len(dropped):
        raise RuntimeError(f"模板结构变化：预期移除 {len(dropped)} 个子图输入，实际 {len(removed)} 个")
    for index in reversed(removed):
        graph["inputs"].pop(index)
    for link in graph["links"]:
        if link["origin_id"] == -10:
            link["origin_slot"] -= sum(1 for index in removed if index < link["origin_slot"])

    # 3) 唯一的参考图放在子图内：LoadImage -> ref_images.ref_image_0。
    left = min([n["pos"][0] for n in graph["nodes"]] + [-2600]) - 400
    top = min([n["pos"][1] for n in graph["nodes"]] + [4700]) + 760
    order = max([n.get("order", 0) for n in graph["nodes"]] + [0]) + 1
    load = add_node({
        "type": "LoadImage", "pos": [left, top], "size": [320, 340],
        "flags": {}, "order": order, "mode": 0, "inputs": [],
        "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": []},
                    {"name": "MASK", "type": "MASK", "links": []}],
        "properties": {"cnr_id": "comfy-core", "ver": "0.33.0", "Node name for S&R": "LoadImage"},
        "widgets_values": ["", "image"], "widgets_values_named": {"image": "", "upload": "image"},
        "title": "Reference image <Picture 1>（参考图，不当首帧）"})
    image_link = add_link(load["id"], 0, cond_id, 0, "IMAGE")
    load["outputs"][0]["links"].append(image_link)

    # 4) ref2va 输入顺序：clip / vae / audio_vae / 参考图 / 提示词 / 尺寸。
    cond["type"] = "MiniMaxH3ReferenceToVideo"
    cond["properties"] = dict(cond.get("properties") or {})
    cond["properties"]["Node name for S&R"] = "MiniMaxH3ReferenceToVideo"
    named = cond.get("widgets_values_named", {})
    width = named.get("width", 1344)
    height = named.get("height", 768)
    length = named.get("length", 124)
    cond["inputs"] = [
        {"name": "clip", "type": "CLIP", "link": old["clip"]},
        {"name": "vae", "type": "VAE", "link": old["vae"]},
        {"name": "audio_vae", "type": "VAE", "link": audio_vae_link},
        {"label": "ref_image_0", "name": "ref_images.ref_image_0", "shape": 7, "type": "IMAGE",
         "link": image_link},
        {"name": "prompt", "type": "STRING", "widget": {"name": "prompt"}, "link": old["prompt"]},
        {"name": "width", "type": "INT", "widget": {"name": "width"}, "link": old["width"]},
        {"name": "height", "type": "INT", "widget": {"name": "height"}, "link": old["height"]},
        {"name": "length", "type": "INT", "widget": {"name": "length"}, "link": old["length"]},
    ]
    # 参考图走 2048 短边（max），并把 turbo 关掉：这两项最影响「像不像」。
    cond["widgets_values"] = [REF_IMAGE_PROMPT, width, height, length, IT2V_REF_IMAGE_SIZE]
    cond["widgets_values_named"] = {"prompt": REF_IMAGE_PROMPT, "width": width, "height": height,
                                    "length": length, "ref_image_size": IT2V_REF_IMAGE_SIZE}
    slots = {inp["link"]: slot for slot, inp in enumerate(cond["inputs"]) if inp.get("link") is not None}
    for link in graph["links"]:
        if link["target_id"] == cond_id:
            if link["id"] not in slots:
                raise RuntimeError(f"模板结构变化：ref2va 有未配置的连线 {link['id']}")
            link["target_slot"] = slots[link["id"]]

    # 5) 模型：ref2va GGUF + ref2v turbo LoRA；fl2va 的档案保持原样。
    dit_url = f"https://huggingface.co/{dit_item['repo']}/resolve/{dit_item['revision']}/{dit_item['file']}"
    lora_url = f"https://huggingface.co/{lora_item['repo']}/resolve/{lora_item['revision']}/{lora_item['file']}"
    for node in graph["nodes"]:
        if node["type"] in ("UnetLoaderGGUF", "UNETLoader"):
            node["widgets_values"] = [MULTI_REF_DIT]
            node["widgets_values_named"] = {"unet_name": MULTI_REF_DIT}
            node["properties"] = {"Node name for S&R": "UnetLoaderGGUF", "models": [
                {"name": MULTI_REF_DIT, "directory": "diffusion_models", "url": dit_url}]}
        elif node["type"] == "LoraLoaderModelOnly":
            strength = node["widgets_values"][1] if len(node["widgets_values"]) > 1 else 1.0
            node["widgets_values"] = [MULTI_REF_LORA, strength]
            node["widgets_values_named"] = {"lora_name": MULTI_REF_LORA, "strength_model": strength}
            node["properties"] = {"Node name for S&R": "LoraLoaderModelOnly", "models": [
                {"name": MULTI_REF_LORA, "directory": "loras", "url": lora_url}]}

    # 5b) 执行期真正喂给加载器的是「促销到子图实例上」的 COMBO widget。
    named_inst = instance.get("widgets_values_named") or {}
    if "unet_name" in named_inst:
        set_widget(instance, "unet_name", MULTI_REF_DIT)
    if "lora_name" in named_inst:
        set_widget(instance, "lora_name", MULTI_REF_LORA)

    # 默认关掉 turbo（走 20 步）：4-step turbo 对参考图的约束力最弱。
    # 实例上的 value 是促销控件；同时把子图里真正接线的那个布尔节点改掉，避免只改一边。
    set_widget(instance, "value", IT2V_TURBO)
    value_slot = next((i for i, inp in enumerate(graph["inputs"]) if inp["name"] == "value"), None)
    turbo_link = next((l["id"] for l in graph["links"]
                       if l["origin_id"] == -10 and l["origin_slot"] == value_slot), None)
    holder = next((n for n in graph["nodes"]
                   if turbo_link is not None and
                   any(i.get("link") == turbo_link for i in n.get("inputs") or [])), None)
    if holder is not None:
        set_widget(holder, "value", IT2V_TURBO)

    # 顶层 ResolutionSelector 决定输出尺寸；0.1 只有约 320px，脸画不出来。
    for node in workflow["nodes"]:
        if node["type"] == "ResolutionSelector":
            set_widget(node, "megapixels", IT2V_MEGAPIXELS)

    # 6) 顶层：移除只给首帧用的 LoadImage 及其 promoted 输入，再修正槽位。
    keep = [i for i, inp in enumerate(instance["inputs"])
            if inp.get("name") not in ("first_frame", "last_frame")]
    removed_slots = [i for i in range(len(instance["inputs"])) if i not in keep]
    dropped_top = [instance["inputs"][i].get("link") for i in removed_slots]
    dropped_top = [link_id for link_id in dropped_top if link_id is not None]
    origins = {l[0]: l[1] for l in workflow["links"]}
    instance["inputs"] = [instance["inputs"][i] for i in keep]
    kept = []
    for link in workflow["links"]:
        if link[0] in dropped_top:
            continue
        if link[3] == instance["id"]:
            link[4] -= sum(1 for slot in removed_slots if slot < link[4])
        kept.append(link)
    workflow["links"] = kept
    doomed = {origins[link_id] for link_id in dropped_top
              if next((n["type"] for n in workflow["nodes"] if n["id"] == origins[link_id]), "") == "LoadImage"}
    if doomed:
        workflow["nodes"] = [n for n in workflow["nodes"] if n["id"] not in doomed]
    prune_dead_nodes(workflow)

    # 7) 提示词、输出前缀与说明。
    set_widget(instance, "prompt", REF_IMAGE_PROMPT)
    for node in workflow["nodes"]:
        if node.get("type") == "SaveVideo":
            set_widget(node, "filename_prefix", "video/MiniMax_H3_IT2V")
        elif node.get("type") == "MarkdownNote" and node.get("title") == "Note: MiniMax H3":
            text = ("## MiniMax H3 · Kaggle IT2V（单图参考，参考图不当首帧）\n\n"
                    "**输入**：双击子图，上传 `Reference image <Picture 1>`；提示词写在子图实例的 `prompt` 上。\n\n"
                    "- 参考图只提供身份 / 外观：第一帧由模型自己产生，**不会**变成参考图本身。\n"
                    "- 默认值都为了「像不像」：`ref_image_size = max`（2048 短边）、0.4 megapixels、"
                    "turbo 关闭（20 步）。要快速预览就把顶部 `IT2V_*` 常量调回 match / 0.1 / True。\n"
                    "- **参考图请用单人、单一视角**（正面半身或全身）。三视图拼一张会被当成版面或"
                    "平均成一张脸，这是人物跑掉最常见的原因。\n"
                    "- 需要多视角请改用 R2V：正面 / 侧面 / 背面分开放 `Picture 1/2/3`。\n"
                    "- 提示词用 `<Picture 1>` 指称参考图，并保留 no split screen / no multiple views，"
                    "压掉多格画面与重复人物。\n"
                    "- 模型换成 ref2va（与 fl2va 并存、互不覆盖）；显存紧可先把 duration 降到 2 秒。\n")
            node["widgets_values"] = [text]
            node["widgets_values_named"] = {"text": text}
        elif node.get("type") == "MarkdownNote" and node.get("title") == "Note: Model Links":
            text = ("## IT2V 用到的权重（与 fl2va 并存）\n\n"
                    f"- DiT：`{MULTI_REF_DIT}`（ref2va，参考图专用）\n"
                    f"- LoRA：`{MULTI_REF_LORA}`（4 step turbo）\n"
                    "- 文本编码器：`qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors`（GPU1）\n"
                    "- VAE：`minimax_h3_video_vae_fp16` / `minimax_h3_audio_vae_fp32`\n\n"
                    "T2V / T2I / IV2V 仍用 `minimax_h3_fl2va_pruned-Q4_K_M.gguf`，两套文件都在 models 目录。\n")
            node["widgets_values"] = [text]
            node["widgets_values_named"] = {"text": text}
    graph["state"]["lastNodeId"] = max(n["id"] for n in graph["nodes"])
    graph["state"]["lastLinkId"] = max(l["id"] for l in graph["links"])
    workflow["last_node_id"] = max(n["id"] for n in workflow["nodes"])
    workflow["last_link_id"] = max(l[0] for l in workflow["links"])
    return workflow


def build_iv2v_workflow(workflow):
    """由已适配的 I2V 模板派生 IV2V：原片 + 人脸参考图 → 换脸短视频。

    只换子图里的条件节点：MiniMaxH3ImageToVideo → MiniMaxH3ReferenceToVideo（ref2va）。
    ref2va 用 <Picture 1> 提供身份、<Video 1> 提供动作，正好是「用影片换脸」；
    顶层用 LoadVideo → GetVideoComponents 取原片帧（和音轨），ImageFromBatch 截取前 N 帧。
    权重、Turbo LoRA、第二张 GPU 上的文本编码器全部沿用，不下载任何新模型。
    """
    workflow = json.loads(json.dumps(workflow))
    subgraphs = workflow.get("definitions", {}).get("subgraphs", [])
    if len(subgraphs) != 1:
        raise RuntimeError(f"模板结构变化：预期 1 个 H3 子图，实际 {len(subgraphs)} 个")
    graph = subgraphs[0]
    if len(graph.get("outputs", [])) != 1 or graph["outputs"][0].get("type") != "VIDEO":
        raise RuntimeError("模板结构变化：H3 子图输出不是单一 VIDEO")
    cond = next((n for n in graph["nodes"] if n["type"] == "MiniMaxH3ImageToVideo"), None)
    if cond is None:
        raise RuntimeError("模板结构变化：子图里缺少 MiniMaxH3ImageToVideo")
    instance = next((n for n in workflow["nodes"] if n["type"] == graph["id"]), None)
    if instance is None:
        raise RuntimeError("模板结构变化：找不到 H3 子图实例节点")
    audio_decode = next((n for n in graph["nodes"] if n["type"] == "VAEDecodeAudio"), None)
    audio_vae_link_in = next((l for l in graph["links"] if audio_decode is not None and
                              l["target_id"] == audio_decode["id"] and l["type"] == "VAE"), None)
    audio_vae = next((n for n in graph["nodes"] if audio_vae_link_in is not None and
                      n["id"] == audio_vae_link_in["origin_id"]), None)
    if audio_vae is None:
        raise RuntimeError("模板结构变化：找不到音频 VAE 加载器")

    old = {i["name"]: i.get("link") for i in cond["inputs"]}
    missing = {"clip", "vae", "first_frame", "last_frame", "prompt", "width", "height", "length"} - set(old)
    if missing:
        raise RuntimeError("模板结构变化：MiniMaxH3ImageToVideo 缺少输入 " + ", ".join(sorted(missing)))

    # 1) 子图 links 是对象格式：先把音频 VAE 接到 ref2va 的 audio_vae，再声明参考音轨输入。
    state = graph.setdefault("state", {})
    next_link = max([state.get("lastLinkId", 0)] + [l["id"] for l in graph["links"]]) + 1
    audio_vae_link = next_link
    audio_ref_link = next_link + 1
    cond_id = cond["id"]
    graph["links"].append({"id": audio_vae_link, "origin_id": audio_vae["id"], "origin_slot": 0,
                           "target_id": cond_id, "target_slot": 2, "type": "VAE"})
    audio_vae["outputs"][0]["links"] = list(audio_vae["outputs"][0].get("links") or []) + [audio_vae_link]

    input_slot = len(graph["inputs"])
    graph["inputs"].append({"id": f"{audio_ref_link:08x}-0000-4000-8000-{audio_ref_link:012x}",
                            "name": "ref_video_audio", "type": "AUDIO",
                            "linkIds": [audio_ref_link], "pos": [-2412.37890625, 5114]})
    graph["links"].append({"id": audio_ref_link, "origin_id": -10, "origin_slot": input_slot,
                           "target_id": cond_id, "target_slot": 5, "type": "AUDIO"})
    graph["name"] = "Reference to Video (MiniMax H3 · IV2V)"
    for inp in graph["inputs"]:
        if inp["name"] == "first_frame":
            inp["label"] = "reference_image (face)"
        elif inp["name"] == "last_frame":
            inp["label"] = "reference_video (frames)"
        elif inp["name"] == "ref_video_audio":
            inp["label"] = "reference_video_audio"

    # 2) ref2va 的输入顺序：clip / vae / audio_vae / ref_image / ref_video / ref_video_audio / 四个控件。
    cond["type"] = "MiniMaxH3ReferenceToVideo"
    cond["properties"] = dict(cond.get("properties") or {})
    cond["properties"]["Node name for S&R"] = "MiniMaxH3ReferenceToVideo"
    named = cond.get("widgets_values_named", {})
    width = named.get("width", 1344)
    height = named.get("height", 768)
    length = named.get("length", 124)
    cond["inputs"] = [
        {"name": "clip", "type": "CLIP", "link": old["clip"]},
        {"name": "vae", "type": "VAE", "link": old["vae"]},
        {"name": "audio_vae", "type": "VAE", "link": audio_vae_link},
        {"label": "ref_image_0", "name": "ref_images.ref_image_0", "shape": 7, "type": "IMAGE",
         "link": old["first_frame"]},
        {"label": "ref_video_0", "name": "ref_videos.ref_video_0", "shape": 7, "type": "IMAGE",
         "link": old["last_frame"]},
        {"label": "ref_video_audio_0", "name": "ref_video_audios.ref_video_audio_0", "shape": 7,
         "type": "AUDIO", "link": audio_ref_link},
        {"name": "prompt", "type": "STRING", "widget": {"name": "prompt"}, "link": old["prompt"]},
        {"name": "width", "type": "INT", "widget": {"name": "width"}, "link": old["width"]},
        {"name": "height", "type": "INT", "widget": {"name": "height"}, "link": old["height"]},
        {"name": "length", "type": "INT", "widget": {"name": "length"}, "link": old["length"]},
    ]
    cond["widgets_values"] = [IV2V_PROMPT, width, height, length, "match"]
    cond["widgets_values_named"] = {"prompt": IV2V_PROMPT, "width": width, "height": height,
                                   "length": length, "ref_image_size": "match"}
    slots = {audio_vae_link: 2, old["first_frame"]: 3, old["last_frame"]: 4, audio_ref_link: 5,
             old["prompt"]: 6, old["width"]: 7, old["height"]: 8, old["length"]: 9}
    for link in graph["links"]:
        if link["target_id"] == cond_id and link["id"] in slots:
            link["target_slot"] = slots[link["id"]]
    state["lastLinkId"] = audio_ref_link

    # 3) 顶层 links 是数组格式：LoadVideo → GetVideoComponents → ImageFromBatch → 子图实例。
    nodes = workflow["nodes"]
    base_id = max([workflow.get("last_node_id", 0)] + [n["id"] for n in nodes])
    link_base = max([workflow.get("last_link_id", 0)] + [l[0] for l in workflow["links"]])
    ref_slot = next(i for i, inp in enumerate(instance["inputs"]) if inp["name"] == "last_frame")
    instance["inputs"][ref_slot].update({"label": "reference_video (frames)", "link": None})
    for inp in instance["inputs"]:
        if inp["name"] == "first_frame":
            inp["label"] = "reference_image (face)"
    audio_slot = len(instance["inputs"])
    load_video_id, components_id, batch_id = base_id + 1, base_id + 2, base_id + 3
    l_video, l_frames, l_trim, l_audio = link_base + 1, link_base + 2, link_base + 3, link_base + 4
    order = max([n.get("order", 0) for n in nodes] + [0])
    nodes.extend([
        {"id": load_video_id, "type": "LoadVideo", "pos": [-2560, 4860], "size": [360, 400],
         "flags": {}, "order": order + 1, "mode": 0, "inputs": [],
         "outputs": [{"name": "VIDEO", "type": "VIDEO", "links": [l_video]}],
         "properties": {"cnr_id": "comfy-core", "ver": "0.33.0", "Node name for S&R": "LoadVideo"},
         "widgets_values": ["", "video"],
         "widgets_values_named": {"file": "", "upload": "video"},
         "title": "Source video (换脸用的原片)"},
        {"id": components_id, "type": "GetVideoComponents", "pos": [-2180, 4860], "size": [300, 170],
         "flags": {}, "order": order + 2, "mode": 0,
         "inputs": [{"name": "video", "type": "VIDEO", "link": l_video}],
         "outputs": [{"name": "images", "type": "IMAGE", "links": [l_frames]},
                     {"name": "audio", "type": "AUDIO", "links": [l_audio]},
                     {"name": "fps", "type": "FLOAT", "links": []},
                     {"name": "bit_depth", "type": "COMBO", "links": []},
                     {"name": "color_space", "type": "COMBO", "links": []}],
         "properties": {"cnr_id": "comfy-core", "ver": "0.33.0", "Node name for S&R": "GetVideoComponents"},
         "title": "Split video into frames + audio"},
        {"id": batch_id, "type": "ImageFromBatch", "pos": [-1820, 4860], "size": [310, 110],
         "flags": {}, "order": order + 3, "mode": 0,
         "inputs": [{"name": "image", "type": "IMAGE", "link": l_frames}],
         "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [l_trim]}],
         "properties": {"cnr_id": "comfy-core", "ver": "0.33.0", "Node name for S&R": "ImageFromBatch"},
         "widgets_values": [0, IV2V_REF_FRAMES],
         "widgets_values_named": {"batch_index": 0, "length": IV2V_REF_FRAMES},
         "title": f"Reference frames (前 {IV2V_REF_FRAMES} 帧 ≈ 5s @24fps)"},
    ])
    workflow["links"].extend([
        [l_video, load_video_id, 0, components_id, 0, "VIDEO"],
        [l_frames, components_id, 0, batch_id, 0, "IMAGE"],
        [l_trim, batch_id, 0, instance["id"], ref_slot, "IMAGE"],
        [l_audio, components_id, 1, instance["id"], audio_slot, "AUDIO"],
    ])
    instance["inputs"][ref_slot]["link"] = l_trim
    instance["inputs"].append({"label": "reference_video_audio", "name": "ref_video_audio",
                               "shape": 7, "type": "AUDIO", "link": l_audio})
    # 3b) 清掉模板自带、不参与执行的辅助支路，并重算最大节点 / 连线编号。
    prune_dead_nodes(workflow)
    workflow["last_node_id"] = max(n["id"] for n in workflow["nodes"])
    workflow["last_link_id"] = max(l[0] for l in workflow["links"])

    # 4) 提示词、输出前缀与上传占位；只用同一套权重。
    set_widget(instance, "prompt", IV2V_PROMPT)
    for node in workflow["nodes"]:
        if node.get("type") == "SaveVideo":
            set_widget(node, "filename_prefix", "video/MiniMax_H3_IV2V")
        elif node.get("type") == "LoadImage":
            set_widget(node, "image", "")
            node["title"] = "Reference face image (换上去的人脸)"
        elif node.get("type") == "MarkdownNote" and node.get("title") == "Note: MiniMax H3":
            text = ("## MiniMax H3 · Kaggle IV2V（换脸）\n\n"
                    "**输入**：`Source video` 上传原片；`Reference face image` 上传要换上去的人脸。\n\n"
                    "**原理**：子图里的条件节点换成了 `MiniMaxH3ReferenceToVideo`（ref2va），"
                    "`<Picture 1>` 给身份、`<Video 1>` 给动作 / 镜位 / 节奏。参考帧默认取原片前 "
                    f"{IV2V_REF_FRAMES} 帧（24fps ≈ 5s），输出时长更短时会自动截断；原片音轨也一并作为参考音讯。\n\n"
                    "**不下载新模型**：沿用 T2V 的同一套 H3 权重、Turbo LoRA 与 GPU1 上的文本编码器。\n\n"
                    "- 换脸是「重新生成」而不是像素级贴脸：背景与细节会被轻微重绘，人脸越清晰效果越好。\n"
                    "- 先在低分辨率 / 2 秒短片测试，成功后再调 duration（子图实例上的滑块）与 megapixels。\n"
                    "- 想更贴脸可进子图把 ref2va 的 `ref_image_size` 由 match 改为 max（慢数倍、更吃显存）。\n"
                    "- 原片没有音轨也能跑；若这一步报错，断开 Get Video Components 到子图 `reference_video_audio` 的连线。\n")
            node["widgets_values"] = [text]
            node["widgets_values_named"] = {"text": text}
    return workflow


def build_multi_ref_workflow(workflow):
    """由已适配的 I2V 模板派生多参考 (ref2va) 工作流。

    3 张参考图 + 2 段参考影片（各自带音轨）+ 提示词，全部交给同一个
    MiniMaxH3ReferenceToVideo 节点。参考素材的 LoadImage / LoadVideo 放在
    子图内，避免改动子图对外的 promotion 接口；顶层仍只有 prompt / 尺寸 /
    duration / turbo 这些控制项。

    模型换成 ref2va pruned Q4_K_M GGUF 与 ref2v turbo 4-step LoRA，
    与 fl2va 并存，不覆盖任何既有文件。
    """
    workflow = json.loads(json.dumps(workflow))
    subgraphs = workflow.get("definitions", {}).get("subgraphs", [])
    if len(subgraphs) != 1:
        raise RuntimeError(f"模板结构变化：预期 1 个 H3 子图，实际 {len(subgraphs)} 个")
    graph = subgraphs[0]
    if len(graph.get("outputs", [])) != 1 or graph["outputs"][0].get("type") != "VIDEO":
        raise RuntimeError("模板结构变化：H3 子图输出不是单一 VIDEO")
    cond = next((n for n in graph["nodes"] if n["type"] == "MiniMaxH3ImageToVideo"), None)
    if cond is None:
        raise RuntimeError("模板结构变化：子图里缺少 MiniMaxH3ImageToVideo")
    instance = next((n for n in workflow["nodes"] if n["type"] == graph["id"]), None)
    if instance is None:
        raise RuntimeError("模板结构变化：找不到 H3 子图实例节点")
    audio_decode = next((n for n in graph["nodes"] if n["type"] == "VAEDecodeAudio"), None)
    audio_vae_link_in = next((l for l in graph["links"] if audio_decode is not None and
                              l["target_id"] == audio_decode["id"] and l["type"] == "VAE"), None)
    audio_vae = next((n for n in graph["nodes"] if audio_vae_link_in is not None and
                      n["id"] == audio_vae_link_in["origin_id"]), None)
    if audio_vae is None:
        raise RuntimeError("模板结构变化：找不到音频 VAE 加载器")

    old = {i["name"]: i.get("link") for i in cond["inputs"]}
    missing = {"clip", "vae", "first_frame", "last_frame", "prompt", "width", "height", "length"} - set(old)
    if missing:
        raise RuntimeError("模板结构变化：MiniMaxH3ImageToVideo 缺少输入 " + ", ".join(sorted(missing)))

    dit_item = next(item for item in MANIFEST if Path(item["file"]).name == MULTI_REF_DIT)
    lora_item = next(item for item in MANIFEST if Path(item["file"]).name == MULTI_REF_LORA)
    state = graph.setdefault("state", {})
    cond_id = cond["id"]
    next_node = max([state.get("lastNodeId", 0)] + [n["id"] for n in graph["nodes"]]) + 1
    next_link = max([state.get("lastLinkId", 0)] + [l["id"] for l in graph["links"]]) + 1

    def add_node(node):
        nonlocal next_node
        node["id"] = next_node
        next_node += 1
        graph["nodes"].append(node)
        return node

    def add_link(origin_id, origin_slot, target_id, target_slot, kind):
        nonlocal next_link
        graph["links"].append({"id": next_link, "origin_id": origin_id, "origin_slot": origin_slot,
                               "target_id": target_id, "target_slot": target_slot, "type": kind})
        next_link += 1
        return next_link - 1

    # 1) 音频 VAE 接上 ref2va 的 audio_vae：参考影片的音轨要用它编码。
    audio_vae_link = add_link(audio_vae["id"], 0, cond_id, 2, "VAE")
    audio_vae["outputs"][0]["links"] = list(audio_vae["outputs"][0].get("links") or []) + [audio_vae_link]

    # 2) 拆掉首尾帧接线：ref2va 没有 first_frame / last_frame。
    dropped = [old["first_frame"], old["last_frame"]]
    dropped = [link_id for link_id in dropped if link_id is not None]
    graph["links"] = [l for l in graph["links"] if l["id"] not in dropped]
    removed = sorted(i for i, inp in enumerate(graph["inputs"])
                     if set(inp.get("linkIds") or []) & set(dropped))
    if len(removed) != 2:
        raise RuntimeError(f"模板结构变化：预期移除 2 个子图输入，实际 {len(removed)} 个")
    for index in reversed(removed):
        graph["inputs"].pop(index)
    for link in graph["links"]:
        if link["origin_id"] == -10:
            link["origin_slot"] -= sum(1 for index in removed if index < link["origin_slot"])

    # 3) 参考素材放在子图内：图 -> ref_images；影片 -> ref_videos + ref_video_audios。
    image_links, video_links, video_audio_links = [], [], []
    order = max([n.get("order", 0) for n in graph["nodes"]] + [0]) + 1
    left = min([n["pos"][0] for n in graph["nodes"]] + [-2600]) - 400
    top = min([n["pos"][1] for n in graph["nodes"]] + [4700]) + 760
    for index in range(MULTI_REF_IMAGES):
        node = add_node({
            "type": "LoadImage", "pos": [left, top + index * 380], "size": [300, 330],
            "flags": {}, "order": order, "mode": 0, "inputs": [],
            "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": []},
                        {"name": "MASK", "type": "MASK", "links": []}],
            "properties": {"cnr_id": "comfy-core", "ver": "0.33.0", "Node name for S&R": "LoadImage"},
            "widgets_values": ["", "image"], "widgets_values_named": {"image": "", "upload": "image"},
            "title": f"Reference image <Picture {index + 1}>"})
        link_id = add_link(node["id"], 0, cond_id, 0, "IMAGE")
        node["outputs"][0]["links"].append(link_id)
        image_links.append(link_id)
        order += 1
    column = left + 380
    for index in range(MULTI_REF_VIDEOS):
        load = add_node({
            "type": "LoadVideo", "pos": [column, top + index * 560], "size": [360, 400],
            "flags": {}, "order": order, "mode": 0, "inputs": [],
            "outputs": [{"name": "VIDEO", "type": "VIDEO", "links": []}],
            "properties": {"cnr_id": "comfy-core", "ver": "0.33.0", "Node name for S&R": "LoadVideo"},
            "widgets_values": ["", "video"], "widgets_values_named": {"file": "", "upload": "video"},
            "title": f"Reference video <Video {index + 1}> (24fps, 2-15s)"})
        parts = add_node({
            "type": "GetVideoComponents", "pos": [column + 400, top + index * 560], "size": [320, 180],
            "flags": {}, "order": order + 1, "mode": 0,
            "inputs": [{"name": "video", "type": "VIDEO", "link": None}],
            "outputs": [{"name": "images", "type": "IMAGE", "links": []},
                        {"name": "audio", "type": "AUDIO", "links": []},
                        {"name": "fps", "type": "FLOAT", "links": []},
                        {"name": "bit_depth", "type": "COMBO", "links": []},
                        {"name": "color_space", "type": "COMBO", "links": []}],
            "properties": {"cnr_id": "comfy-core", "ver": "0.33.0", "Node name for S&R": "GetVideoComponents"}})
        batch = add_node({
            "type": "ImageFromBatch", "pos": [column + 760, top + index * 560], "size": [320, 110],
            "flags": {}, "order": order + 2, "mode": 0,
            "inputs": [{"name": "image", "type": "IMAGE", "link": None}],
            "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": []}],
            "properties": {"cnr_id": "comfy-core", "ver": "0.33.0", "Node name for S&R": "ImageFromBatch"},
            "widgets_values": [0, MULTI_REF_REF_FRAMES],
            "widgets_values_named": {"batch_index": 0, "length": MULTI_REF_REF_FRAMES},
            "title": f"Trim <Video {index + 1}> to {MULTI_REF_REF_FRAMES} frames"})
        video_link = add_link(load["id"], 0, parts["id"], 0, "VIDEO")
        load["outputs"][0]["links"].append(video_link)
        parts["inputs"][0]["link"] = video_link
        frames_link = add_link(parts["id"], 0, batch["id"], 0, "IMAGE")
        parts["outputs"][0]["links"].append(frames_link)
        batch["inputs"][0]["link"] = frames_link
        trim_link = add_link(batch["id"], 0, cond_id, 0, "IMAGE")
        batch["outputs"][0]["links"].append(trim_link)
        audio_link = add_link(parts["id"], 1, cond_id, 0, "AUDIO")
        parts["outputs"][1]["links"].append(audio_link)
        video_links.append(trim_link)
        video_audio_links.append(audio_link)
        order += 3

    # 4) ref2va 输入顺序：clip / vae / audio_vae / 图 / 影片 / 影片音轨 / 四个控件。
    cond["type"] = "MiniMaxH3ReferenceToVideo"
    cond["properties"] = dict(cond.get("properties") or {})
    cond["properties"]["Node name for S&R"] = "MiniMaxH3ReferenceToVideo"
    named = cond.get("widgets_values_named", {})
    width = named.get("width", 1344)
    height = named.get("height", 768)
    length = named.get("length", 124)
    cond["inputs"] = [
        {"name": "clip", "type": "CLIP", "link": old["clip"]},
        {"name": "vae", "type": "VAE", "link": old["vae"]},
        {"name": "audio_vae", "type": "VAE", "link": audio_vae_link},
    ]
    for index, link_id in enumerate(image_links):
        cond["inputs"].append({"label": f"ref_image_{index}", "name": f"ref_images.ref_image_{index}",
                               "shape": 7, "type": "IMAGE", "link": link_id})
    for index, link_id in enumerate(video_links):
        cond["inputs"].append({"label": f"ref_video_{index}", "name": f"ref_videos.ref_video_{index}",
                               "shape": 7, "type": "IMAGE", "link": link_id})
    for index, link_id in enumerate(video_audio_links):
        cond["inputs"].append({"label": f"ref_video_audio_{index}",
                               "name": f"ref_video_audios.ref_video_audio_{index}",
                               "shape": 7, "type": "AUDIO", "link": link_id})
    cond["inputs"] += [
        {"name": "prompt", "type": "STRING", "widget": {"name": "prompt"}, "link": old["prompt"]},
        {"name": "width", "type": "INT", "widget": {"name": "width"}, "link": old["width"]},
        {"name": "height", "type": "INT", "widget": {"name": "height"}, "link": old["height"]},
        {"name": "length", "type": "INT", "widget": {"name": "length"}, "link": old["length"]},
    ]
    cond["widgets_values"] = [MULTI_REF_PROMPT, width, height, length, "match"]
    cond["widgets_values_named"] = {"prompt": MULTI_REF_PROMPT, "width": width, "height": height,
                                    "length": length, "ref_image_size": "match"}
    slots = {inp["link"]: slot for slot, inp in enumerate(cond["inputs"]) if inp.get("link") is not None}
    for link in graph["links"]:
        if link["target_id"] == cond_id:
            if link["id"] not in slots:
                raise RuntimeError(f"模板结构变化：ref2va 有未配置的连线 {link['id']}")
            link["target_slot"] = slots[link["id"]]

    # 5) 模型：ref2va GGUF + ref2v turbo LoRA；fl2va 的档案保持原样。
    dit_url = f"https://huggingface.co/{dit_item['repo']}/resolve/{dit_item['revision']}/{dit_item['file']}"
    lora_url = f"https://huggingface.co/{lora_item['repo']}/resolve/{lora_item['revision']}/{lora_item['file']}"
    for node in graph["nodes"]:
        if node["type"] in ("UnetLoaderGGUF", "UNETLoader"):
            node["widgets_values"] = [MULTI_REF_DIT]
            node["widgets_values_named"] = {"unet_name": MULTI_REF_DIT}
            node["properties"] = {"Node name for S&R": "UnetLoaderGGUF", "models": [
                {"name": MULTI_REF_DIT, "directory": "diffusion_models", "url": dit_url}]}
        elif node["type"] == "LoraLoaderModelOnly":
            strength = node["widgets_values"][1] if len(node["widgets_values"]) > 1 else 1.0
            node["widgets_values"] = [MULTI_REF_LORA, strength]
            node["widgets_values_named"] = {"lora_name": MULTI_REF_LORA, "strength_model": strength}
            node["properties"] = {"Node name for S&R": "LoraLoaderModelOnly", "models": [
                {"name": MULTI_REF_LORA, "directory": "loras", "url": lora_url}]}

    # 5b) 执行期真正喂给加载器的是「促销到子图实例上」的 COMBO widget：
    #     子图里 unet_name / lora_name 已被输入连线接管，只改节点 widget 会被覆盖。
    named_inst = instance.get("widgets_values_named") or {}
    if "unet_name" in named_inst:
        set_widget(instance, "unet_name", MULTI_REF_DIT)
    if "lora_name" in named_inst:
        set_widget(instance, "lora_name", MULTI_REF_LORA)

    # 6) 顶层：移除只给 first_frame 用的 LoadImage 及其连线，并修正 promoted 槽位。
    keep = [i for i, inp in enumerate(instance["inputs"])
            if inp.get("name") not in ("first_frame", "last_frame")]
    removed_slots = [i for i in range(len(instance["inputs"])) if i not in keep]
    dropped_top = [instance["inputs"][i].get("link") for i in removed_slots]
    dropped_top = [link_id for link_id in dropped_top if link_id is not None]
    origins = {l[0]: l[1] for l in workflow["links"]}
    instance["inputs"] = [instance["inputs"][i] for i in keep]
    kept = []
    for link in workflow["links"]:
        if link[0] in dropped_top:
            continue
        if link[3] == instance["id"]:
            link[4] -= sum(1 for slot in removed_slots if slot < link[4])
        kept.append(link)
    workflow["links"] = kept
    doomed = {origins[link_id] for link_id in dropped_top
              if next((n["type"] for n in workflow["nodes"] if n["id"] == origins[link_id]), "") == "LoadImage"}
    if doomed:
        workflow["nodes"] = [n for n in workflow["nodes"] if n["id"] not in doomed]

    # 6b) 清掉模板自带、不参与执行的辅助支路（如未接线的 ImageScaleToTotalPixels / GetImageSize）。
    prune_dead_nodes(workflow)

    # 7) 提示词、输出前缀与说明。
    set_widget(instance, "prompt", MULTI_REF_PROMPT)
    for node in workflow["nodes"]:
        if node.get("type") == "SaveVideo":
            set_widget(node, "filename_prefix", "video/MiniMax_H3_R2V")
        elif node.get("type") == "MarkdownNote" and node.get("title") == "Note: MiniMax H3":
            text = ("## MiniMax H3 · Kaggle R2V（多图 + 多影 + 提示词）\n\n"
                    "**参考素材**（都在子图内，双击子图即可上传）：\n"
                    f"- `Reference image <Picture 1..{MULTI_REF_IMAGES}>`：{MULTI_REF_IMAGES} 张参考图。\n"
                    f"- `Reference video <Video 1..{MULTI_REF_VIDEOS}>`：{MULTI_REF_VIDEOS} 段参考影片，各取前 "
                    f"{MULTI_REF_REF_FRAMES} 帧（24fps 约 {MULTI_REF_REF_FRAMES / 24:.1f} 秒），"
                    "影片自带的音轨会一并当参考音频。\n\n"
                    "**提示词**用 `<Picture i>` / `<Video k>` / `<Audio j>` 指称素材；`<Audio j>` 的编号跟着"
                    "影片音轨（第 1 段影片的音轨 = `<Audio 1>`）。\n\n"
                    "**模型**：`minimax_h3_ref2va_pruned-Q4_K_M.gguf` + `minimax_h3_ref2v_turbo_4step` LoRA。"
                    "参考图 / 影片要真的生效需要 ref2va 权重，它和 fl2va 是不同的 checkpoint；"
                    "T2V / T2I / IV2V 仍用原本的 fl2va，两套文件并存、互不覆盖。\n\n"
                    "**显存**：参考 latent 每一步都参与，T4 14.5GB 很紧。先维持 0.1 megapixels、2 秒短片，"
                    "成功后再调 duration / megapixels。\n\n"
                    "- `ref_image_size`：`match` 会把参考图缩到生成解析度的面积（省显存、identity 较弱）；"
                    "子图内改成 `max` 会用 2048 短边（最贴脸，但慢数倍、很吃显存）。\n"
                    "- 参考影片不足 5 帧或超过 15 秒都会被截断；非 24fps 的影片会被当成 24fps 使用。\n"
                    "- 参考影片没有音轨也能跑；若这一步报错，把该影片的 `audio` 连线断开。\n")
            node["widgets_values"] = [text]
            node["widgets_values_named"] = {"text": text}
        elif node.get("type") == "MarkdownNote" and node.get("title") == "Note: Model Links":
            text = ("## R2V 用到的权重（与 fl2va 并存）\n\n"
                    f"- DiT：`{MULTI_REF_DIT}`（ref2va，多参考专用）\n"
                    f"- LoRA：`{MULTI_REF_LORA}`（4 step turbo）\n"
                    "- 文本编码器：`qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors`（GPU1）\n"
                    "- VAE：`minimax_h3_video_vae_fp16` / `minimax_h3_audio_vae_fp32`\n\n"
                    "T2V / T2I / IV2V 仍用 `minimax_h3_fl2va_pruned-Q4_K_M.gguf`，IT2V（单图参考）用 ref2va，"
                    "两套文件都在 models 目录，互不覆盖。\n")
            node["widgets_values"] = [text]
            node["widgets_values_named"] = {"text": text}
    graph["state"]["lastNodeId"] = max(n["id"] for n in graph["nodes"])
    graph["state"]["lastLinkId"] = max(l["id"] for l in graph["links"])
    workflow["last_node_id"] = max(n["id"] for n in workflow["nodes"])
    workflow["last_link_id"] = max(l[0] for l in workflow["links"])
    return workflow


print("1/6 检查实际 GPU、内存和旧服务...", flush=True)
probe = subprocess.run([sys.executable, "-c", "import json,torch; print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'gpus':[{'name':torch.cuda.get_device_name(i),'vram':torch.cuda.get_device_properties(i).total_memory,'capability':torch.cuda.get_device_capability(i)} for i in range(torch.cuda.device_count())]}))"], capture_output=True, text=True)
if probe.returncode:
    raise RuntimeError("PyTorch / CUDA 检查失败：" + probe.stderr[-3000:])
hardware = json.loads(probe.stdout.strip().splitlines()[-1])
if not hardware["gpus"]:
    raise RuntimeError("未检测到 CUDA GPU。先在 Kaggle Settings > Accelerator 选择 GPU T4 x2，并重新启动会话。")
for i, gpu in enumerate(hardware["gpus"]):
    print(f"GPU {i}: {gpu['name']} / {gpu['vram']/2**30:.2f} GiB", flush=True)
second_gpu = USE_SECOND_GPU and len(hardware["gpus"]) >= 2
print("设备分配：", "DiT/VAE → GPU0；文本编码器 → GPU1" if second_gpu else "单卡 / 自动卸载模式", flush=True)
print("双卡总显存不是自动合并的显存池；本脚本显式给文本编码器指定 GPU1。", flush=True)
mem = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
print(f"系统内存：{int(mem['MemTotal'].split()[0])/2**20:.1f} GiB；当前可用 {int(mem['MemAvailable'].split()[0])/2**20:.1f} GiB", flush=True)
if int(mem['MemAvailable'].split()[0]) < 24 * 2**20:
    print("⚠️ 当前可用系统内存不足 24 GiB，H3 仍可能内存不足；请停止占用资源的模型服务。", flush=True)
stop_owned("comfy_process")
with socket.socket() as sock:
    if sock.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"端口 {PORT} 被其他服务占用。请停止旧 ComfyUI/Ollama 网关，或更改 PORT；没有自动杀死未知进程。")

print("2/6 更新 ComfyUI 和 H3 专用 GGUF 插件...", flush=True)
ROOT.parent.mkdir(parents=True, exist_ok=True)
clone_or_update("https://github.com/Comfy-Org/ComfyUI.git", ROOT)
plugin = ROOT / "custom_nodes/ComfyUI-GGUF"
if (plugin / ".git").is_dir():
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=plugin, text=True).strip()
    if "leejet/ComfyUI-GGUF" not in origin:
        backup = ROOT / "disabled_custom_nodes" / f"ComfyUI-GGUF-{int(time.time())}"
        backup.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(plugin), str(backup))
        print("已保留旧 GGUF 插件备份：", backup, flush=True)
clone_or_update("https://github.com/leejet/ComfyUI-GGUF.git", plugin)

print("3/6 安装依赖，保留 Kaggle 自带 CUDA PyTorch...", flush=True)
venv = ROOT.parent / "h3-python-env"
python = prepare_python_env(venv, sys.executable)
# 在安装其他依赖前确认 venv 仍可见 Kaggle 的 CUDA PyTorch。
torch_probe = subprocess.run([str(python), "-c",
    "import torch; assert torch.cuda.is_available(); print(torch.__version__)"],
    capture_output=True, text=True)
if torch_probe.returncode:
    raise RuntimeError("虚拟环境无法继承 Kaggle CUDA PyTorch：\n" + torch_probe.stderr[-4000:])
if torch_probe.stdout.strip().splitlines()[-1] != hardware["torch"]:
    raise RuntimeError("虚拟环境 PyTorch 版本与 Notebook 不同，请改用新的 h3-python-env 目录后重跑。")
constraints = ROOT / "h3-torch-constraints.txt"
constraints.write_text("torch==" + hardware["torch"] + "\n")
run(python, "-m", "pip", "install", "-q", "-c", constraints, "-r", ROOT / "requirements.txt", "-r", plugin / "requirements.txt")
check = subprocess.run([str(python), "-c", "import torch; assert torch.cuda.is_available(); import gguf; import av; print(torch.__version__)"], capture_output=True, text=True)
if check.returncode:
    raise RuntimeError("依赖 / CUDA 自检失败：" + check.stderr[-4000:])
print("PyTorch：", check.stdout.strip(), flush=True)

print("4/6 下载 H3 全套权重，支持断点续传和 SHA-256 校验...", flush=True)
needed = 0
for item in MANIFEST:
    target = ROOT / "models" / item["folder"] / Path(item["file"]).name
    if target.exists() and target.stat().st_size == item["size"]:
        continue
    part = target.with_name(target.name + ".part")
    partial = min(part.stat().st_size, item["size"]) if part.exists() else 0
    needed += item["size"] - partial
free = shutil.disk_usage(ROOT).free
print(f"模型合计 {sum(x['size'] for x in MANIFEST)/2**30:.2f} GiB；剩余下载约 {needed/2**30:.2f} GiB；可用磁盘 {free/2**30:.2f} GiB", flush=True)
if free < needed + 3 * 2**30:
    raise RuntimeError("磁盘不足：请为剩余下载和输出视频至少再留 3 GiB；不会自动删除现有模型。")
for item in MANIFEST:
    download_model(item)

print("5/6 保存官方 H3 模板的双卡 GGUF 适配副本，启动服务...", flush=True)
templates = json.loads(zlib.decompress(base64.b64decode(TEMPLATE_BUNDLE)))
workflow_dir = ROOT / "user/default/workflows"
workflow_dir.mkdir(parents=True, exist_ok=True)
patched_workflows = {}
for name, workflow in templates.items():
    workflow = patch_workflow(workflow, second_gpu)
    patched_workflows[name] = workflow
    if name == "i2v":
        # IT2V 要改成单图参考（ref2va），统一由 build_ref_image_workflow 写出。
        continue
    path = workflow_dir / f"MiniMax_H3_Kaggle_{WORKFLOW_NAMES.get(name, name.upper())}.json"
    path.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding="utf-8")
    print("工作流：", path, flush=True)
it2v_workflow = build_ref_image_workflow(patched_workflows["i2v"])
it2v_path = workflow_dir / "MiniMax_H3_Kaggle_IT2V.json"
it2v_path.write_text(json.dumps(it2v_workflow, ensure_ascii=False, indent=2), encoding="utf-8")
print("工作流（单图参考，参考图不当首帧）：", it2v_path, flush=True)
server_log = ROOT / "comfyui-h3.log"
command = [str(python), "-u", "main.py", "--listen", "127.0.0.1", "--port", str(PORT),
           "--cuda-device", "all", "--disable-pinned-memory", "--cache-none", "--preview-method", "none"]
# 不强制 fp16/bf16，避免 T4 上 H3 超范围激活变 NaN；使用模型自身精度策略。
with server_log.open("a", encoding="utf-8") as log:
    comfy_process = subprocess.Popen(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
deadline = time.monotonic() + 240
while not ready():
    if comfy_process.poll() is not None:
        raise RuntimeError("ComfyUI 启动失败：\n" + tail(server_log))
    if time.monotonic() >= deadline:
        raise TimeoutError("ComfyUI 启动超时：\n" + tail(server_log))
    time.sleep(1)
objects = request_json("/object_info", timeout=60)
required = {"MiniMaxH3ImageToVideo", "UnetLoaderGGUF", "CLIPLoader", "VAELoader", "SaveVideo", "SaveImage",
            "ResolutionSelector"}
if second_gpu:
    required.add("SelectCLIPDevice")
missing = required - objects.keys()
if missing:
    raise RuntimeError("缺少节点：" + ", ".join(sorted(missing)) + "\n" + tail(server_log))
clip_spec = objects["CLIPLoader"]["input"]["required"]["type"]
clip_types = clip_spec[0] if isinstance(clip_spec[0], list) else (
    clip_spec[1].get("options") if len(clip_spec) > 1 and isinstance(clip_spec[1], dict) else None)
if clip_types is not None and "minimax" not in clip_types:
    raise RuntimeError("CLIPLoaderGGUF 没有 minimax 类型，请更新 ComfyUI 和 leejet 插件。")
if second_gpu:
    spec = objects["SelectCLIPDevice"]["input"]["required"]["device"]
    # 旧版：[选项列表, 元数据]；新版：["COMBO", {"options": 选项列表}]。
    choices = spec[0] if isinstance(spec[0], list) else (
        spec[1].get("options") if len(spec) > 1 and isinstance(spec[1], dict) else None)
    if choices is None:
        print("设备选项未在接口中展开；已检测到两张 GPU，保留工作流 gpu:1 设置。", flush=True)
    elif "gpu:1" not in choices:
        raise RuntimeError("设备节点实际选项中没有 gpu:1：" + str(choices))
    else:
        print("第二张 GPU 的设备选项检查通过：gpu:1", flush=True)
t2i_workflow = build_t2i_workflow(patched_workflows["t2v"], "ImageFromBatch" in objects)
t2i_path = workflow_dir / "MiniMax_H3_Kaggle_T2I.json"
t2i_path.write_text(json.dumps(t2i_workflow, ensure_ascii=False, indent=2), encoding="utf-8")
print("工作流：", t2i_path, flush=True)
if "ImageFromBatch" not in objects:
    print("⚠️ 缺少 ImageFromBatch：T2I 会保存整段 5 帧 PNG，而不是只留第一帧。", flush=True)
iv2v_required = {"MiniMaxH3ReferenceToVideo", "LoadVideo", "GetVideoComponents", "ImageFromBatch"}
iv2v_missing = iv2v_required - objects.keys()
if iv2v_missing:
    print("⚠️ 当前 ComfyUI 缺少节点，跳过 IV2V：", ", ".join(sorted(iv2v_missing)), flush=True)
else:
    iv2v_workflow = build_iv2v_workflow(patched_workflows["i2v"])
    iv2v_path = workflow_dir / "MiniMax_H3_Kaggle_IV2V.json"
    iv2v_path.write_text(json.dumps(iv2v_workflow, ensure_ascii=False, indent=2), encoding="utf-8")
    print("工作流：", iv2v_path, flush=True)
r2v_required = {"MiniMaxH3ReferenceToVideo", "LoadVideo", "GetVideoComponents", "ImageFromBatch"}
r2v_missing = r2v_required - objects.keys()
if r2v_missing:
    print("⚠️ 当前 ComfyUI 缺少节点，跳过 R2V 多参考：", ", ".join(sorted(r2v_missing)), flush=True)
else:
    r2v_workflow = build_multi_ref_workflow(patched_workflows["i2v"])
    r2v_path = workflow_dir / "MiniMax_H3_Kaggle_R2V.json"
    r2v_path.write_text(json.dumps(r2v_workflow, ensure_ascii=False, indent=2), encoding="utf-8")
    print("工作流：", r2v_path, flush=True)
print("节点和服务就绪；尚未执行视频生成，启动成功不代表推理自检通过。", flush=True)

print("6/6 创建 / 复用 Cloudflare 临时访问链接...", flush=True)
if START_TUNNEL:
    old_tunnel = globals().get("tunnel_process")
    old_url = globals().get("url", "")
    if not (old_tunnel is not None and old_tunnel.poll() is None and
            globals().get("_h3_tunnel_target") == LOCAL_URL and
            re.fullmatch(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", str(old_url))):
        stop_owned("tunnel_process")
        binary = ROOT.parent / "cloudflared"
        if not binary.exists():
            temporary = binary.with_suffix(".download")
            source = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            with urllib.request.urlopen(source, timeout=120) as response, temporary.open("wb") as f:
                shutil.copyfileobj(response, f)
            if temporary.read_bytes()[:4] != b"\x7fELF":
                raise RuntimeError("cloudflared 下载内容不是 Linux ELF 文件")
            temporary.replace(binary)
        binary.chmod(0o755)
        tunnel_log = ROOT / "cloudflared-h3.log"
        with tunnel_log.open("w", encoding="utf-8") as log:
            tunnel_process = subprocess.Popen([str(binary), "tunnel", "--url", LOCAL_URL, "--no-autoupdate"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        deadline = time.monotonic() + 90
        while True:
            content = tail(tunnel_log)
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                url = match.group(0)
                _h3_tunnel_target = LOCAL_URL
                break
            if tunnel_process.poll() is not None or time.monotonic() >= deadline:
                raise RuntimeError("隧道启动失败：\n" + content)
            time.sleep(0.5)
    print("ComfyUI 访问网址：", url, flush=True)
else:
    print("ComfyUI 本地地址：", LOCAL_URL, flush=True)
print("打开 Workflows，加载 MiniMax_H3_Kaggle_T2V、MiniMax_H3_Kaggle_IT2V、MiniMax_H3_Kaggle_T2I、MiniMax_H3_Kaggle_IV2V 或 MiniMax_H3_Kaggle_R2V。")
print("IT2V 单图参考：双击子图上传 Reference image <Picture 1>；默认 max 参考图 / 0.4MP / 20 步，参考图请用单张正面。")
print("T2I 不额外下载模型：复用同一套 H3 权重采最短 5 帧，取第一帧存 PNG。")
print("IV2V 换脸：Load Video 上传原片、Load Image 上传人脸图；改用 ref2va 节点把脸换成图片里的人，同一套权重不额外下载。")
print("IV2V 参考帧默认取原片前 124 帧（24fps 约 5s），输出时长改子图实例上的 duration。")
print("内置模板库中的原始 H3 模板仍使用 safetensors：请优先使用本脚本生成的适配副本，避免重复下载大模型。")
print(f"R2V 多参考：子图内上传 {MULTI_REF_IMAGES} 张图 + {MULTI_REF_VIDEOS} 段影片；模型改用 ref2va Q4_K_M（与 fl2va 并存）。")
print("视频目录：", ROOT / "output/video")
print("图片目录：", ROOT / "output/image")
print("诊断日志：", server_log)
print("临时链接可访问此 ComfyUI，会话关闭后失效；不要把链接公开发布。")


def stop_services():
    stop_owned("tunnel_process")
    stop_owned("comfy_process")
    print("本脚本的 ComfyUI 和隧道已停止。")


1/6 检查实际 GPU、内存和旧服务...
GPU 0: Tesla T4 / 14.56 GiB
GPU 1: Tesla T4 / 14.56 GiB
设备分配： DiT/VAE → GPU0；文本编码器 → GPU1
双卡总显存不是自动合并的显存池；本脚本显式给文本编码器指定 GPU1。
系统内存：31.3 GiB；当前可用 29.8 GiB
2/6 更新 ComfyUI 和 H3 专用 GGUF 插件...


Cloning into '/tmp/ComfyUI'...
Cloning into '/tmp/ComfyUI/custom_nodes/ComfyUI-GGUF'...


3/6 安装依赖，保留 Kaggle 自带 CUDA PyTorch...
pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
PyTorch： 2.11.0+cu130
4/6 下载 H3 全套权重，支持断点续传和 SHA-256 校验...
模型合计 32.48 GiB；剩余下载约 32.48 GiB；可用磁盘 965.05 GiB
下载 minimax_h3_fl2va_pruned-Q4_K_M.gguf，已完成 0.00 GiB，尝试 1/3
  3.34 / 10.64 GiB
  6.95 / 10.64 GiB
校验 SHA-256： minimax_h3_fl2va_pruned-Q4_K_M.gguf.part
下载 qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors，已完成 0.00 GiB，尝试 1/3
  3.07 / 14.61 GiB
  5.02 / 14.61 GiB
  8.07 / 14.61 GiB
  10.54 / 14.61 GiB
  13.70 / 14.61 GiB
校验 SHA-256： qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors.part
下载 minimax_h3_video_vae_fp16.safetensors，已完成 0.00 GiB，尝试 1/3
  3.84 / 4.85 GiB
校验 SHA-256： minimax_h3_video_vae_fp16.safetensors.part
下载 minimax_h3_audio_vae_fp32.safetensors，已完成 0.00 GiB，尝试 1/3
校验 SHA-256： minimax_h3_audio_vae_fp32.safetensors.part
下载 minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors，已完成 0.00 GiB，尝试 1/3
校验 SHA-256： minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safete

In [ ]:
# ==================== Keepalive：避免互动 session 因闲置被 Kaggle 关闭 ====================
# 用法：主 cell 跑完并印出 ComfyUI 网址后，执行这个 cell（保持执行中即可）。
# 原理：cell 一直处于执行中，kernel 不算闲置，20~60 分钟的 idle timeout 就不会触发。
# 结束：按该 cell 的停止钮，然后执行 stop_services()。
# 注意：只挡闲置逾时，挡不了 12 小时的互动 session 硬上限。

KEEPALIVE_SECONDS = 60        # 心跳间隔（秒）
KEEPALIVE_MAX_HOURS = 11.0    # 主动收工，留 1 小时缓冲避开 12 小时硬上限
KEEPALIVE_FAIL_LIMIT = 3      # ComfyUI 连续失联几次才判定挂掉

import time

_started = time.monotonic()
_deadline = _started + KEEPALIVE_MAX_HOURS * 3600
_fails = 0

print(f"[keepalive] 启动，上限 {KEEPALIVE_MAX_HOURS} 小时；停止本 cell 才会结束。", flush=True)
try:
    while True:
        time.sleep(KEEPALIVE_SECONDS)
        _elapsed = time.monotonic() - _started
        try:
            _alive = ready()
        except Exception:
            _alive = False

        _fails = 0 if _alive else _fails + 1
        print("[keepalive] %s 已维持 %.2f h | ComfyUI=%s%s"
              % (time.strftime("%H:%M:%S"), _elapsed / 3600,
                 "OK" if _alive else "DOWN",
                 "" if _alive else f"（连续 {_fails}/{KEEPALIVE_FAIL_LIMIT}）"),
              flush=True)

        if _fails >= KEEPALIVE_FAIL_LIMIT:
            print("[keepalive] ComfyUI 连续失联，结束 keepalive。请看 comfyui-h3.log 后重跑主 cell。", flush=True)
            break
        if time.monotonic() >= _deadline:
            print(f"[keepalive] 已达 {KEEPALIVE_MAX_HOURS} 小时，主动停止（12 小时硬上限保护）。", flush=True)
            break
except KeyboardInterrupt:
    print("[keepalive] 已中断。记得执行 stop_services() 关闭 ComfyUI 与隧道。", flush=True)
